In [11]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

X = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/train.csv', index_col='id') 
X_test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e9/test.csv', index_col='id')

X.dropna(axis=0, subset=['Will_Buy_EV'], inplace=True)

y = X['Will_Buy_EV'].map({'Yes': 1, 'No': 0})

X.drop(['Will_Buy_EV'], axis=1, inplace=True)

X.head()

,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level
id,,,,,,,,,,,,,
0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low
1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low
2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low
3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low
4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low


In [15]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Identify categorical and numerical columns directly from X
categorical_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']
numerical_cols = X.select_dtypes(exclude=['object', 'bool']).columns.tolist()

# Numerical preprocessing
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# Feature engineering
## Added variables
- Total charging stations (sum of at home and work)
- Income per car owned
- Commute km relative to income
- Environment concerns amplified possibilities by income

Accuracy stayed the same.

In [24]:
def add_features(df):
    df = df.copy()
    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['Income_per_Car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)
    df['Commute_Income_Ratio'] = df['Daily_Commute_km'] / (df['Annual_Income_USD'] + 1) * 10000
    df['Environment_Incom_Multiplication'] = df['Environmental_Concern_Level'] * (df['Annual_Income_USD'] + 1)
    return df

X = add_features(X)
X_test = add_features(X_test)

Results without XGBoost: 

```
Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best CV Accuracy: 0.8949690519096359
Best Parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 300}
```

## XGBoost

Results with XGBoost:
```
Fitting 3 folds for each of 63 candidates, totalling 189 fits
Best CV Accuracy: 0.8984409239938813
Best Parameters: {'model__learning_rate': 0.03, 'model__max_depth': 6, 'model__n_estimators': 800}
```

# Ensembling
## Add LightGBM

In [34]:
from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import cross_val_score

# best_xgb_model = grid_search.best_estimator_

best_xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=6,
        random_state=0,
        n_jobs=2,
        eval_metric='logloss'
    ))
])


lgbm_model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    random_state=0,
    n_jobs=2
)

lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', lgbm_model)
])

# Combine XGBoost and LightGBM
ensemble_model = VotingClassifier(
    estimators=[
        ('xgb', best_xgb_model),
        ('lgb', lgbm_pipeline)
    ],
    voting='soft'
)

cv_scores = cross_val_score(ensemble_model, X, y, cv=3, scoring='accuracy', n_jobs=-1)
print(f"Ensemble CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

ensemble_model.fit(X, y)

ensemble_predictions = ensemble_model.predict(X_test)
submission_ensemble = pd.DataFrame({
    'id': X_test.index,
    'Will_Buy_EV': ensemble_predictions
})
submission_ensemble.to_csv('submission_ensemble.csv', index=False)
print("Saved Ensemble Submission")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Ensemble CV Accuracy: 0.8979 (+/- 0.0003)
[LightGBM] [Info] Number of positive: 77852, number of negative: 367924
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.075857 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 636
[LightGBM] [Info] Number of data points in the train set: 445776, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174644 -> initscore=-1.553067
[LightGBM] [Info] Start training from score -1.553067
[LightGBM] [Info] Number of positive: 116779, number of negative: 551886
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027426 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 636
[LightGBM] [Info] Number of data points in the train set: 668665, n

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Ensemble Submission erfolgreich gespeichert!


## Add CatBoost

In [36]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=800,
    learning_rate=0.03,
    depth=6,
    random_seed=0,
    verbose=0,
    thread_count=2
)

cat_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', cat_model)
])

ensemble_model_3 = VotingClassifier(
    estimators=[
        ('xgb', best_xgb_model),
        ('lgb', lgbm_pipeline),
        ('cat', cat_pipeline)
    ],
    voting='soft',
    weights=[1.2, 1.0, 1.0]
)

ensemble_model_3.fit(X, y)

chunks = np.array_split(X_test, 5)
predictions_list_3 = []

for i, chunk in enumerate(chunks):
    chunk_pred = ensemble_model_3.predict(chunk)
    predictions_list_3.append(chunk_pred)

ensemble_predictions_3 = np.concatenate(predictions_list_3)

submission_ensemble_3 = pd.DataFrame({
    'id': X_test.index,
    'Will_Buy_EV': ensemble_predictions_3
})
submission_ensemble_3.to_csv('submission_ensemble_3_models.csv', index=False)

[LightGBM] [Info] Number of positive: 116779, number of negative: 551886
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027603 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 636
[LightGBM] [Info] Number of data points in the train set: 668665, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.174645 -> initscore=-1.553058
[LightGBM] [Info] Start training from score -1.553058


/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/l